# Part 1: Foundation Model Exploration

## Overview

This notebook applies a pre-trained multilingual sentiment model (HuggingFace) to Olist customer review text and compares its performance to the custom Random Forest model built in HW2.

**Key distinction:**
- **HW2 model:** Predicts satisfaction from order/delivery features *before* the review is written (proactive early-warning system)
- **Foundation model:** Analyzes review text *after* it's written (reactive classification tool)

These serve different business purposes. The goal is to explore what foundation models can do and understand the tradeoffs.


In [1]:
!pip -q install transformers torch

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

Setup complete!


## 1A. Setup and Inference

We load the Olist dataset, filter to records with non-empty review text, sample 500 records, and run the pre-trained multilingual sentiment model.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Load Olist dataset with review text
df = pd.read_csv("/content/drive/MyDrive/olist_data/olist_cleaned_hw1.csv")
print("Dataset shape:", df.shape)
print("review_comment_message present:", "review_comment_message" in df.columns)

Mounted at /content/drive
Dataset shape: (119143, 44)
review_comment_message present: True


In [3]:
# Filter to records with non-empty review text
df_reviews = df[
    df["review_comment_message"].notna() &
    (df["review_comment_message"].str.strip() != "")
].copy()
print(f"Records with review text: {len(df_reviews)}")

# Create binary target (same definition as HW2)
df_reviews["is_positive_review"] = np.where(
    df_reviews["review_score"].isin([4, 5]), 1, 0
)

# Sample 500 records
sample = df_reviews.sample(n=500, random_state=42).reset_index(drop=True)
print(f"Sample size: {len(sample)}")
print(f"\nActual class distribution:")
print(sample["is_positive_review"].value_counts().rename(
    {1: "Positive (4-5 stars)", 0: "Negative (1-3 stars)"}
))


Records with review text: 118115
Sample size: 500

Actual class distribution:
is_positive_review
Positive (4-5 stars)    370
Negative (1-3 stars)    130
Name: count, dtype: int64


In [4]:
# Load pre-trained multilingual sentiment model
sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=0  # Use GPU
)

print("Model loaded! Running inference on 500 records...")

# Run predictions
results = sentiment(
    sample["review_comment_message"].tolist(),
    truncation=True,
    max_length=512
)

# Extract predicted star ratings and map to binary
sample["fm_label"] = [r["label"] for r in results]
sample["fm_score"] = [r["score"] for r in results]
sample["fm_stars"] = sample["fm_label"].str.extract(r"(\d)").astype(int)
sample["fm_positive"] = np.where(sample["fm_stars"] >= 4, 1, 0)

print("\nFoundation model predicted class distribution:")
print(sample["fm_positive"].value_counts().rename(
    {1: "Predicted Positive", 0: "Predicted Negative"}
))

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Model loaded! Running inference on 500 records...

Foundation model predicted class distribution:
fm_positive
Predicted Negative    408
Predicted Positive     92
Name: count, dtype: int64


The table above shows how many records are actually positive vs. negative in the 500-record sample, and how many the foundation model predicted as positive vs. negative. This gives us an initial sense of the model's classification behavior before computing formal metrics.

## 1B. Performance Comparison

We compare the foundation model's predictions against the HW2 custom Random Forest model, both evaluated on the same 500-record sample. To make this a fair comparison, we retrain the HW2 model and generate its predictions on the same records.

In [5]:
# Engineer features that may be missing from the sample
if "total_cost" not in sample.columns and "price" in sample.columns:
    sample["total_cost"] = sample["price"] + sample["freight_value"]
if "log_price" not in sample.columns and "price" in sample.columns:
    sample["log_price"] = np.log1p(sample["price"])
if "is_late" not in sample.columns and "delivery_vs_estimated" in sample.columns:
    sample["is_late"] = (sample["delivery_vs_estimated"] > 0).astype(int)

# Retrain HW2 model for fair comparison
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

hw2_features = [
    "delivery_days", "delivery_vs_estimated", "price", "freight_value",
    "seller_state", "payment_type", "payment_installments", "payment_value",
    "product_weight_g", "product_length_cm", "product_height_cm",
    "product_width_cm", "customer_state", "total_cost", "log_price", "is_late"
]

df_hw2 = pd.read_csv("/content/drive/MyDrive/olist_data/hw2_prepared_dataset.csv")
X_hw2 = df_hw2[hw2_features]
y_hw2 = df_hw2["is_positive_review"]
X_train, X_test, y_train, y_test = train_test_split(
    X_hw2, y_hw2, test_size=0.2, stratify=y_hw2, random_state=42
)

num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

preprocess = ColumnTransformer([
    ("num", SkPipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),
    ("cat", SkPipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

hw2_model = SkPipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=50, max_depth=15,
        min_samples_split=5, random_state=42
    ))
])
hw2_model.fit(X_train, y_train)
print("HW2 model retrained.")

HW2 model retrained.


In [6]:
# Generate HW2 predictions on the same 500 records
X_sample = sample[hw2_features]
y_hw2_pred = hw2_model.predict(X_sample)

y_true = sample["is_positive_review"]
y_fm = sample["fm_positive"]

# Build comparison table
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Foundation Model (review text)": [
        accuracy_score(y_true, y_fm),
        precision_score(y_true, y_fm),
        recall_score(y_true, y_fm),
        f1_score(y_true, y_fm)
    ],
    "HW2 Model (order features)": [
        accuracy_score(y_true, y_hw2_pred),
        precision_score(y_true, y_hw2_pred),
        recall_score(y_true, y_hw2_pred),
        f1_score(y_true, y_hw2_pred)
    ]
})

# Round for display
comparison.iloc[:, 1:] = comparison.iloc[:, 1:].round(4)
comparison

,Metric,Foundation Model (review text),HW2 Model (order features)
0,Accuracy,0.4080,0.7900
1,Precision,0.9022,0.7813
2,Recall,0.2243,0.9946
3,F1 Score,0.3593,0.8751


## 1C. Reflection

### Which model performed better?

The HW2 custom Random Forest model substantially outperformed the foundation model across almost every metric. The HW2 model achieved an F1 of 0.875 compared to the foundation model's 0.359. While the foundation model showed high precision (0.902), its recall was extremely low (0.224), meaning it correctly identified only about 22% of actual positive reviews. The HW2 model, by contrast, achieved near-perfect recall (0.995) — catching virtually every positive case.

The foundation model's poor performance likely stems from two factors: (1) the multilingual sentiment model was not fine-tuned on e-commerce review data and may interpret Portuguese text differently than domain-specific patterns require, and (2) the model's star-rating predictions tend to skew conservative, classifying many genuinely positive reviews as neutral (3 stars), which maps to "negative" under our binary threshold.

### Advantages and disadvantages of zero-training approach

**Advantages:**
- No labeled training data required — works out of the box
- Immediate deployment with zero data science effort
- Handles multilingual text (including Portuguese) without custom training
- No risk of overfitting to a specific dataset

**Disadvantages:**
- No domain-specific tuning — as demonstrated here, generic sentiment models can perform poorly on specialized e-commerce data
- Requires the review to already exist — cannot be used proactively
- Computationally expensive (transformer inference vs. Random Forest)
- Less interpretable — harder to explain individual predictions to stakeholders
- The 0.224 recall shows the model misses the vast majority of positive reviews, making it unreliable for business decisions

### When to use foundation model vs. custom model?

A foundation model is ideal when labeled data is unavailable, the task aligns closely with the model's pre-training objective, and speed-to-deployment matters. However, as this experiment demonstrates, when sufficient domain-specific labeled data exists — as it does for Olist — a custom-trained model significantly outperforms a generic foundation model. The custom model benefits from features engineered with domain knowledge (delivery timing, pricing signals) that capture business-specific satisfaction drivers the text model cannot access.

### Could these approaches be combined?

Yes, but the combination would need to account for the foundation model's limitations. The HW2 model should remain the **primary early-warning system**, flagging at-risk orders before delivery. The foundation model could serve as a **supplementary signal** — not as a standalone classifier, but as an additional feature. For example, the foundation model's confidence score could be fed as an input feature into an enhanced version of the HW2 model, enriching predictions with text-derived sentiment when review text becomes available. This would leverage the foundation model's high precision (0.902) while relying on the custom model's superior overall classification ability.